# Distancia entre secuencias y árboles filogenéticos 

Un árbol filogenético es la representación de una hipótesis de **relaciones ancestrales**, basada en **similitud**. En esta representación, las especies o secuencias de interés estan en las puntas (*tips*) de las ramas. El nodo donde se unen las ramas representa el ancestro común. El tiempo por lo tanto va desde el ancestro común hacia las puntas, en la siguiente figura el tiempo sería el eje horizontal x  

<img src="Anatomy_tree.png"/>


## Inferencia Filogenética

Para cuantificar la **similitud** calculamos las distancias (diferencias) entre grupos y, basado en esta **matriz de distancia**, se decide el orden de agrupamiento (*cluster*). Esta matriz es calculada a partir de una medida de similitud (por ejemplo entre morfologías o un alineamiento). Este primer agrupamiento no garantiza una solución óptima, así que en segundo lugar se busca el mejor árbol (pero esta parte la omitiremos).
Vamos a comenzar con el cálculo de la matriz de distancia apartir de un alineamiento, y a partir de estas distancias, generar el árbol.

### Generación de un alineamiento entre secuencias

Dentro de R también se puede realizar alineamiento entre secuencias, si bien no es recomendable secuencias de larga longitud. Para esto se deben instalar y cargar las librerías necesarias para trabajar con secuencias, así como lo hicimos con python

In [ ]:
# Load the 'Biostrings' package. 
# 'Biostrings' provides efficient containers to handle and analyze biological sequences.
library(Biostrings)
# Load the phangorn library: Provides tools for phylogenetic reconstruction and analysis
library(phangorn)
# Load the ape library: Essential for reading, writing, and manipulating phylogenetic trees
library(ape)

# Define un vector que contiene 4 secuencias de nucleótidos
sequences = c("ACTGGCTG", "ACTGCTG", "AGTGACT", "TGTGACTGA")

# Convy]ierte el vector de secuencias en un objeto DNAStringSet usando Biostrings
biostrings_sequences = DNAStringSet(sequences)

# Realiza un alineamiento múltiple sobre el objeto DNAStringSet usando 'msa' 
msa_result_sample = msa(biostrings_sequences,method = "ClustalW")

# Observe el resultado del alineamiento
print(msa_result_sample)

# Y otro ejemplo con amino ácidos con diferentes longitudes 
# y usando la matríz de sustitución Blosum

aa_sequences <- AAStringSet(c(
 "MKVLYRLSIK",
 "MKILYKLSIK",
 "MKIYYKLSIR",
 "GKVFYKLNIK",
 "AKVFYKLNIK"
))

# Realiza la alineación múltiple de las secuencias de aminoácidos.
# El argumento 'type' se establece en "protein" para indicar que estamos trabajando con secuencias de proteínas.
amino_sequences_aln = msa(aa_sequences, type = "protein",substitutionMatrix = "blosum")
print(amino_sequences_aln)

### Medidas de distancia

Existen diferentes formas de calcula una distancia entre dos puntos, entre las más comunes están (tomado de https://medium.com/@varshamanideepa/exploring-the-different-types-of-distance-measures-for-data-analysis-f5be9133ac3e):
- **Hamming**: cuantifica la diferencia entre dos vectores binarios 

<img src="hamming_dist.webp"  width=150 height=150/> 

- **Euclídea**: d(x,y) = √(∑(xi-yi)²)

<img src="euclidean_dist.webp" width=150 height=150/>

- **Manhattan**: d(x,y) = ∑|xi-yi|

<img src="manhattan_dist.webp" width=150 height=150/>


Con el alineamiento realizado, vamos a obtener la matriz de ditancia

### Modelos de sustitución

Para un grupo de secuencias, la distancia se puede calular apartir del alineamiento. Esta distancia es una medida cuantitativa de la divergencia entre secuencias, y para algunas secuencias (aquellas en las que las tasas de evolución son aproximadamente similares), sirve como estimación del tiempo que ha pasado entre esta divergencia (llamado el *reloj molecular*). 

Para distancias entre secuencias utilizamos modelos de evolución que cuantifican la probabilidad de cambio:

- K80: El modelo Kimura asume diferencias entre las transiciones y transversiones con un parámetro

-  JC: El model Jukes-Cantor asume igual frecuencia entre bases y en tasas de mutación

- WAG: Modelo para amino ácidos

para una mejor comprensión de modelos: http://www.iqtree.org/doc/Substitution-Models

In [ ]:

# Convertir el alineamiento multiple en un objeto phyDat para análisis downstream en phangorn
phyDat_aa = as.phyDat(amino_sequences_aln)

# Se asignan nombres a las secuencias
names(phyDat_aa) = c("Seq 1", "Seq 2", "Seq 3", "Seq 4", "Seq 5")

#Diferentes formas de calcular distancia
aa_sample_hamming = dist.hamming(phyDat_aa, ratio = FALSE) #distancia Hamming

aa_sample_wag <- dist.ml(phyDat_aa, model="WAG") #modelo evolutivo wag

<div class="alert alert-block alert-info">
<b>Compare las dos distancias y piense cómo agruparía las secuencias?</b> 
</div>


### UPGMA: Unweighted Pair Group Method with Arithmetic mean

El método UPGMA es un ejemplo de cluster jerárquico. La distancia entre dos clusters se calcula con la media de las distancias entre cada punto del primer cluster al segundo cluster

En el siguiente link puede encontrar un ejemplo paso a paso para construir el siguiente agrupamiento: http://www.slimsuite.unsw.edu.au/teaching/upgma/

<img src="upgma15.png"/>

### NJ: Neighbour Joining

Mientras que en un árbol realizado con UPGMA todos los puntos están alineados y las ramas tienen la misma distancia (porque las distancias se calculan por la media), en NJ se calcula la “distancia acumulada” para cada secuencia i (A-E),  $U_i = Σ d_{ij}$. Usando esta distancia acumulada Ui, se calcula una matriz de distancia $e_{ij}$ (para cada par de de secuencias i ay j), de acuerdo a la fórmula 

$e_{ij}$ = $d_{ij}$– $\frac{U_i + U_j}{N – 2}$

<img src="NJfig.png"/>

Para una explicación manual: https://www.tenderisthebyte.com/blog/2022/08/31/neighbor-joining-trees/

Ahora agrupamos las secuencias con los dos métodos que hemos descrito aquí, UPGMA y NJ, y graficamos el árbol

In [ ]:
#Agrupamiento con NJ
aa_upgma <- upgma(aa_sample_wag)

# Agrupamiento de secuencias con UPGMA
aa_nj <- NJ(aa_sample_wag)

# graficamos los dos árboles
par(mfrow = c(2, 1))
plot(aa_upgma, no.margin = TRUE)
plot(aa_nj, no.margin = TRUE)

## Ejercicio con las secuencias de Citocromo B

También es posible importar un árbol realizado para graficarlo desde R. Esto se hace con frecuencia, ya que si bien R tiene la fortaleza de graficar, es mas lento y menos eficiente para realizar los alineamientos y cálculos de distancias. 

A continuación vamos a importar el alineamiento de las secuencias de proteína de Citocromo B que habíamos generado con clustal (archivo .phy). A partir de este archivo vamos a:
- 1. Calcular las distancias por pares 
- 2. Aplicar los algoritmos de agrupamiento
- 3. Graficar el árbol 

In [ ]:
cytB <- read.aa("CytBProt.phy")
cytB_phy_dat <- as.phyDat(cytB)
cytB_dist <- dist.ml(cytB_phy_dat)
cytB_UPGMA <- upgma(cytB_dist)
cytB_NJ <- NJ(cytB_dist)
plot(cytB_UPGMA)
plot(cytB_NJ)

<div class="alert alert-block alert-info">
<b>Ejercicio</b> 
Repita el mismo proceso de alineamiento y generación de árbol con el archivo de nucleótidos de Citocromo B (https://raw.githubusercontent.com/lauraalazar/BiologiaComputacional/main/CytBDNA.txt). Ve alguna diferencia con respecto a las proteínas, por ejemplo en la topología del árbol?
</div>